## 1. Download RAGBench Datasets

This step downloads the FinQA and TAT-QA subsets from [RAGBench](https://huggingface.co/datasets/galileo-ai/ragbench).

The official training, validation, and test splits are preserved without additional splitting. All original columns are saved in Parquet format, along with two metadata columns (`dataset` and `split`).

A `dataset_summary.csv` file is also generated to record the number of examples and columns in each split.

In [43]:
from datasets import load_dataset
from pathlib import Path
import pandas as pd

# Configuration
DATASET_NAME = "galileo-ai/ragbench"
SUBSETS = {"finqa": "FinQA", "tatqa": "TATQA"}
SPLITS = ["train", "validation", "test"]

# Find or create the Claimcheck project directory
current_dir = Path.cwd().resolve()

PROJECT_DIR = next(
    (p for p in [current_dir, *current_dir.parents]
     if p.name.lower() == "claimcheck"),
    current_dir / "Claimcheck"
)

OUTPUT_DIR = PROJECT_DIR / "Data" / "RAGBench"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

summary = []

# Download official splits and save complete datasets
for subset, folder_name in SUBSETS.items():
    dataset = load_dataset(DATASET_NAME, subset)
    folder = OUTPUT_DIR / folder_name
    folder.mkdir(parents=True, exist_ok=True)

    for split in SPLITS:
        df = dataset[split].to_pandas()
        df["dataset"] = subset
        df["split"] = split

        path = folder / f"{split}.parquet"
        df.to_parquet(path, index=False)

        summary.append({
            "dataset": subset,
            "split": split,
            "num_examples": len(df),
            "num_columns": len(df.columns)
        })

        # Verify saved file and row count
        assert path.exists()
        assert len(pd.read_parquet(path)) == len(df)

        print(f"Saved: {subset}/{split} ({len(df)} rows)")

# Save dataset summary
summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    OUTPUT_DIR / "dataset_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nAll RAGBench datasets downloaded successfully!")
print(summary_df.to_string(index=False))

Output directory: /Users/stephanie/Documents/python/Claimcheck/Data/RAGBench
Saved: finqa/train (12502 rows)
Saved: finqa/validation (1766 rows)
Saved: finqa/test (2294 rows)
Saved: tatqa/train (26430 rows)
Saved: tatqa/validation (3336 rows)
Saved: tatqa/test (3338 rows)

All RAGBench datasets downloaded successfully!
dataset      split  num_examples  num_columns
  finqa      train         12502           28
  finqa validation          1766           28
  finqa       test          2294           28
  tatqa      train         26430           28
  tatqa validation          3336           28
  tatqa       test          3338           28


## 2. Column Selection and CSV Export

This step selects 12 relevant columns from the original Parquet files and exports six CSV files for ClaimCheck.

Nested data structures, such as document sentences and support annotations, are serialized as JSON strings to make them easier to read and reconstruct.

The original Parquet files remain unchanged. No examples are removed, and the official split assignments are preserved.

**Note:** Ground truth annotations and reference explanations are retained for evaluation but should not be used as model inputs.

In [44]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Configuration
DATASETS = ["FinQA", "TATQA"]
SPLITS = ["train", "validation", "test"]

# Find or create the Claimcheck project directory
current_dir = Path.cwd().resolve()

PROJECT_DIR = next(
    (p for p in [current_dir, *current_dir.parents]
     if p.name.lower() == "claimcheck"),
    current_dir / "Claimcheck"
)

BASE_DIR = PROJECT_DIR / "Data" / "RAGBench"
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Retain 12 columns for ClaimCheck
KEEP_COLUMNS = [
    "id",
    "question",
    "documents",
    "response",
    "documents_sentences",
    "response_sentences",
    "sentence_support_information",
    "unsupported_response_sentence_keys",
    "overall_supported_explanation",
    "relevance_explanation",
    "dataset",
    "split"
]

# Convert nested structures into JSON strings
def to_python(value):
    if isinstance(value, np.ndarray):
        return to_python(value.tolist())
    if isinstance(value, (list, tuple)):
        return [to_python(v) for v in value]
    if isinstance(value, dict):
        return {str(k): to_python(v) for k, v in value.items()}
    if isinstance(value, np.generic):
        return value.item()
    return value

def serialize(value):
    if isinstance(value, (np.ndarray, list, tuple, dict)):
        return json.dumps(to_python(value), ensure_ascii=False)
    return value

summary = []

# Process all six files without changing official splits
for dataset in DATASETS:
    for split in SPLITS:
        folder = BASE_DIR / dataset
        source = folder / f"{split}.parquet"
        output = folder / f"{split}.csv"

        if not source.exists():
            raise FileNotFoundError(
                f"Missing {source}. Run Cell 1 first."
            )

        df = pd.read_parquet(source)

        missing = set(KEEP_COLUMNS) - set(df.columns)
        if missing:
            raise ValueError(f"Missing columns: {missing}")

        cleaned_df = df[KEEP_COLUMNS].copy()

        for col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].apply(serialize)

        cleaned_df.to_csv(
            output,
            index=False,
            encoding="utf-8-sig"
        )

        # Verify row count, columns, and split assignments
        saved = pd.read_csv(output, keep_default_na=False)

        assert len(saved) == len(df)
        assert list(saved.columns) == KEEP_COLUMNS
        assert saved["split"].eq(split).all()
        assert (
            saved["id"].astype(str).tolist()
            == cleaned_df["id"].astype(str).tolist()
        )

        summary.append({
            "dataset": dataset,
            "split": split,
            "rows": len(saved),
            "columns": len(saved.columns)
        })

        print(f"Completed: {dataset}/{split} ({len(saved)} rows)")

# Display final summary
print("\nFinal Summary:")
print(pd.DataFrame(summary).to_string(index=False))
print("\nAll six CSV files processed successfully!")

Completed: FinQA/train (12502 rows)
Completed: FinQA/validation (1766 rows)
Completed: FinQA/test (2294 rows)
Completed: TATQA/train (26430 rows)
Completed: TATQA/validation (3336 rows)
Completed: TATQA/test (3338 rows)

Final Summary:
dataset      split  rows  columns
  FinQA      train 12502       12
  FinQA validation  1766       12
  FinQA       test  2294       12
  TATQA      train 26430       12
  TATQA validation  3336       12
  TATQA       test  3338       12

All six CSV files processed successfully!


## 3. FinanceBench Data Preparation

This step downloads the 150 publicly available FinanceBench examples from Hugging Face and separates them into two evaluation sets.

- **Test Set (147 examples):** Reserved for final model evaluation.
- **Blind Test (3 examples):** Held out separately for the final demonstration.

The split is reproducible using a fixed random seed. All original columns, including questions, answers, and supporting evidence, are preserved.

FinanceBench is used only for final evaluation, not for training, validation, or threshold selection.

In [45]:
from pathlib import Path
import pandas as pd
import json
import urllib.request

# Configuration
REPO_URL = "https://raw.githubusercontent.com/patronus-ai/financebench/main/data"
RANDOM_SEED = 42
BLIND_TEST_SIZE = 3

# Find or create the project directory
current_dir = Path.cwd().resolve()
PROJECT_DIR = next(
    (p for p in [current_dir, *current_dir.parents]
     if p.name.lower() == "claimcheck"),
    current_dir / "Claimcheck"
)

DATA_DIR = PROJECT_DIR / "Data"
FINANCE_DIR = DATA_DIR / "FinanceBench"
BLIND_DIR = DATA_DIR / "Blind test answers"

FINANCE_DIR.mkdir(parents=True, exist_ok=True)
BLIND_DIR.mkdir(parents=True, exist_ok=True)

# Download the original FinanceBench JSONL files
FILES = [
    "financebench_open_source.jsonl",
    "financebench_document_information.jsonl"
]

for filename in FILES:
    destination = FINANCE_DIR / filename
    urllib.request.urlretrieve(
        f"{REPO_URL}/{filename}",
        destination
    )
    print(f"Downloaded: {filename}")

# Load the official question and document metadata
df = pd.read_json(
    FINANCE_DIR / "financebench_open_source.jsonl",
    lines=True
)

df_meta = pd.read_json(
    FINANCE_DIR / "financebench_document_information.jsonl",
    lines=True
)

assert len(df) == 150
assert df["financebench_id"].is_unique

# Select three reproducible blind test examples
# Fixed blind test examples shared by all team members
BLIND_TEST_IDS = [
    "financebench_id_06655",
    "financebench_id_00005",
    "financebench_id_00080"
]

blind_df = df[
    df["financebench_id"].isin(BLIND_TEST_IDS)
].copy()

test_df = df[
    ~df["financebench_id"].isin(BLIND_TEST_IDS)
].copy()

assert len(blind_df) == 3
assert len(test_df) == 147

test_df["split"] = "test"
blind_df["split"] = "blind_test"

# Preserve original row order
test_df = test_df.sort_index().reset_index(drop=True)
blind_df = blind_df.sort_index().reset_index(drop=True)

# Save nested evidence as valid JSON strings in CSV
def serialize(value):
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value

def save_csv(data, path):
    output = data.copy()
    for column in output.columns:
        output[column] = output[column].apply(serialize)
    output.to_csv(path, index=False, encoding="utf-8-sig")

save_csv(test_df, FINANCE_DIR / "financebench_test.csv")
save_csv(blind_df, BLIND_DIR / "financebench_blind_test.csv")

# Verify split sizes and prevent overlap
assert len(test_df) == 147
assert len(blind_df) == 3
assert set(test_df["financebench_id"]).isdisjoint(
    set(blind_df["financebench_id"])
)

print("\nFinanceBench preparation completed!")
print(f"Test: {len(test_df)} examples")
print(f"Blind test: {len(blind_df)} examples")

print("\nSelected blind test examples:")
print(
    blind_df[
        ["financebench_id", "company", "question"]
    ].to_string(index=False)
)

Downloaded: financebench_open_source.jsonl
Downloaded: financebench_document_information.jsonl

FinanceBench preparation completed!
Test: 147 examples
Blind test: 3 examples

Selected blind test examples:
      financebench_id company                                                                                                                                                                                                                                                                                                                                                           question
financebench_id_06655  Amazon What is Amazon's FY2017 days payable outstanding (DPO)? DPO is defined as: 365 * (average accounts payable between FY2016 and FY2017) / (FY2017 COGS + change in inventory between FY2016 and FY2017). Round your answer to two decimal places. Address the question by using the line items and information shown within the balance sheet and the P&L statement.
financebench_id_00005 Cor


# ClaimCheck: Inserted Error Set (400 Examples)

This notebook generates 400 financial verification claims using SEC annual 10-K filings. The dataset contains 300 general financial claims and 100 Storyboard B claims. Both subsets are generated in the same workflow, with separate company assignments to reduce data leakage and prevent company overlap.

### Dataset Split

| Dataset | Training | Validation | Testing | Total |
|---|---:|---:|---:|---:|
| General financial claims | 210 | 60 | 30 | 300 |
| Storyboard B | 30 | 20 | 50 | 100 |
| **Combined** | **240** | **80** | **80** | **400** |

The combined split is 60% training, 20% validation, and 20% testing.

The 300 general claims use 30 companies, with 21 assigned to training, 6 to validation, and 3 to testing.

Storyboard B uses 10 different companies, with 3 assigned to training, 2 to validation, and 5 to testing.

All 40 companies are assigned to exactly one subset and one split. No company is shared between General Claims and Storyboard B or across training, validation, and testing.

### Data Sources

The code uses the six existing RAGBench CSV files, FinanceBench metadata, and SEC annual 10-K filings.

FinanceBench and RAGBench are used to screen candidate companies for potential overlap. Companies identified as overlapping are excluded from selection.

General financial claims use SEC XBRL financial facts, including revenue, net income, assets, liabilities, and other financial metrics.

Storyboard B uses segment revenue information from SEC annual filings. Segment revenue values are compared against company total revenue for the corresponding reporting period.

SEC source snapshots and company assignments are saved to support reproducibility.

### General Financial Claims

The notebook generates 300 general financial claims from 30 companies, with 10 claims per company.

The generated claims include supported financial statements and inserted errors involving numerical values, reporting periods, financial metrics, and units.

Each claim includes its label, error type, source information, financial values, and supporting SEC evidence references.

### Storyboard B

Storyboard B generates 100 financial verification claims from 10 companies that are not included in the General Claims subset.

The primary focus is segment revenue confusion, where a financial value reported for one business segment is incorrectly presented as the total revenue of the company.

The dataset also includes correct segment revenue claims to evaluate whether verification methods can distinguish valid segment statements from scope errors.

Storyboard B contains 80 Unsupported claims and 20 Supported claims.

The validation split contains 20 Storyboard B examples, and the test split contains 50, satisfying the planned minimum of 40 test examples for the segment revenue evaluation slice.

Financial values must be derived from SEC filing evidence rather than invented source figures.

### Company Selection and Data Leakage Prevention

The notebook selects 40 distinct companies after screening against FinanceBench and RAGBench.

Company assignments are fixed before final claim generation.

The program checks that:

1. General Claims and Storyboard B do not share companies.
2. Training, validation, and testing do not share companies.
3. Each company belongs to exactly one dataset subset and one split.
4. Generated claims do not contain duplicate claim text.
5. Companies identified as overlapping with FinanceBench or RAGBench are excluded.

Company screening reduces the risk of data leakage but does not guarantee that all semantic overlap has been eliminated.

### Outputs

The notebook produces the following datasets and reproducibility records:

- `general_claims_300.csv`: 300 general financial claims.
- General training, validation, and testing CSV files.
- `storyboard_b_100.csv`: 100 generated Storyboard B claims.
- Storyboard B training, validation, and testing CSV files.
- `inserted_error_set_400.csv`: combined dataset containing all 400 claims.
- Combined training, validation, and testing CSV files.
- Locked company assignments for all 40 companies.
- `source_snapshots/`: saved SEC source data and filings.
- Reproducibility manifest and SHA256 checksums.

### Reproducibility

The notebook uses a fixed random seed and deterministic company assignment and claim generation procedures.

Source snapshots, input file hashes, company assignments, and output checksums are recorded to support independent regeneration.

The final dataset should be regenerated using the same source snapshots and input files to verify that the output hashes remain unchanged.

### Validation

The code checks:

1. Exactly 300 General Claims and 100 Storyboard B claims are generated.
2. The combined dataset contains 400 unique claims.
3. General Claims follow the 210/60/30 split.
4. Storyboard B follows the 30/20/50 split.
5. All 40 companies are distinct and remain within their assigned splits.
6. Storyboard B segment revenue values are compared with company total revenue.
7. Required financial evidence is available for generated claims.
8. Source and output checksums are recorded.

If sufficient financial evidence is unavailable, the program stops rather than fabricating claims or publishing an incomplete dataset.

Automated checks do not replace final verification of source evidence. The generated dataset should be audited against the original SEC filings before being treated as validated ground truth.


In [2]:
# Original ClaimCheck dataset generation code.
# Preserved for documentation and future regeneration.
# Do not run unless a new dataset version is intentionally required.

ENABLE_REGENERATION = True

if not ENABLE_REGENERATION:
    raise RuntimeError(
        "Dataset regeneration is disabled. "
        "Use the reproduction notebook to verify the existing 400 claims. "
        "Set ENABLE_REGENERATION = True only if you intend to regenerate the dataset."
    )

from pathlib import Path
from collections import OrderedDict
from datetime import date
import hashlib, json, os, random, re, time, shutil, zipfile, tempfile
import pandas as pd
import requests

# Screen 40 disjoint companies and generate 300 general plus 100 storyboard claims.
# Commit this notebook, locked_company_split.csv, and source_snapshots to GitHub.
SEED = 42
SPLIT_COMPANIES = OrderedDict([('train', 21), ('validation', 6), ('test', 3)])
STORYBOARD_COUNTS = {'train': 30, 'validation': 20, 'test': 50}
QUOTAS = OrderedDict([('Supported', 5), ('Small Numerical', 1), ('Digit or Sign', 1), ('Temporal', 1), ('Metric Confusion', 1), ('Unit', 1)])
EMAIL = os.environ.get('SEC_CONTACT_EMAIL', '').strip()

cwd = Path.cwd().resolve()
PROJECT = next((p for p in (cwd, *cwd.parents) if p.name.casefold() == 'claimcheck'), None)
if PROJECT is None:
    raise RuntimeError('Open the notebook inside the Claimcheck project directory.')
DATA = PROJECT / 'Data'
OUT = DATA / 'Inserted Error Set'
CACHE = OUT / 'source_snapshots'
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
MANIFEST = OUT / 'locked_company_split_40.csv'
if not MANIFEST.exists():
    raise FileNotFoundError('Existing locked_company_split_40.csv is required. Company selection must not be repeated.')
STORY_COMPANIES = {'train': 3, 'validation': 2, 'test': 5}
STORY_ERROR_PER_COMPANY = 5
STORY_SUPPORTED_PER_COMPANY = 5

RAG = DATA / 'RAGBench'
FINANCE_META = DATA / 'FinanceBench' / 'financebench_document_information.jsonl'

# Original candidates, extended deterministically when fewer than 40 qualify.
CANDIDATES = {
    'AAPL':'0000320193','MSFT':'0000789019','GOOGL':'0001652044','AMZN':'0001018724',
    'META':'0001326801','NVDA':'0001045810','AMD':'0000002488','INTC':'0000050863',
    'IBM':'0000051143','ORCL':'0001341439','ADBE':'0000796343','CRM':'0001108524',
    'CSCO':'0000858877','QCOM':'0000804328','TXN':'0000097476','AVGO':'0001730168',
    'MU':'0000723125','AMAT':'0000006951','LRCX':'0000707549','KLAC':'0000319201',
    'ADI':'0000006281','INTU':'0000896878','NOW':'0001373715','PANW':'0001327567',
    'CRWD':'0001535527','SNOW':'0001640147','PLTR':'0001321655','UBER':'0001543151',
    'ABNB':'0001559720','PYPL':'0001633917','V':'0001403161','MA':'0001141391',
    'JPM':'0000019617','BAC':'0000070858','WMT':'0000104169','COST':'0000909832',
    'HD':'0000354950','LOW':'0000060667','KO':'0000021344','PEP':'0000077476',
    'MCD':'0000063908','NKE':'0000320187','DIS':'0001744489','NFLX':'0001065280',
    'T':'0000732717','VZ':'0000732712','GE':'0000040545','CAT':'0000018230',
    'BA':'0000012927','HON':'0000773840'
}
ALIASES = {
    'AAPL':['Apple'], 'MSFT':['Microsoft'], 'GOOGL':['Alphabet','Google'],
    'AMZN':['Amazon'], 'META':['Meta Platforms','Facebook'], 'NVDA':['NVIDIA'],
    'AMD':['Advanced Micro Devices'], 'INTC':['Intel'], 'IBM':['International Business Machines'],
    'ORCL':['Oracle'], 'ADBE':['Adobe'], 'CRM':['Salesforce'], 'CSCO':['Cisco'],
    'QCOM':['Qualcomm'], 'TXN':['Texas Instruments'], 'AVGO':['Broadcom'],
    'MU':['Micron Technology'], 'AMAT':['Applied Materials'], 'LRCX':['Lam Research'],
    'KLAC':['KLA Corporation'], 'ADI':['Analog Devices'], 'INTU':['Intuit'],
    'NOW':['ServiceNow'], 'PANW':['Palo Alto Networks'], 'CRWD':['CrowdStrike'],
    'SNOW':['Snowflake'], 'PLTR':['Palantir'], 'UBER':['Uber'], 'ABNB':['Airbnb'],
    'PYPL':['PayPal'], 'V':['Visa'], 'MA':['Mastercard'],
    'JPM':['JPMorgan Chase','JP Morgan Chase'], 'BAC':['Bank of America'],
    'WMT':['Walmart'], 'COST':['Costco'], 'HD':['Home Depot'],
    'LOW':["Lowe's",'Lowes'], 'KO':['Coca Cola','Coca-Cola'], 'PEP':['PepsiCo'],
    'MCD':["McDonald's",'McDonalds'], 'NKE':['Nike'], 'DIS':['Disney','Walt Disney'],
    'NFLX':['Netflix'], 'T':['AT&T'], 'VZ':['Verizon'], 'GE':['General Electric'],
    'CAT':['Caterpillar'], 'BA':['Boeing'], 'HON':['Honeywell']
}
PDF_ALIASES = {'adobe':'Adobe','amcor':'Amcor','block':'Block',
               'bostonproperties':'Boston Properties','footlocker':'Foot Locker',
               'ultabeauty':'Ulta Beauty'}
METRICS = OrderedDict([
    ('revenue', ('revenue', ['RevenueFromContractWithCustomerExcludingAssessedTax',
                            'RevenueFromContractWithCustomerIncludingAssessedTax','Revenues','SalesRevenueNet'])),
    ('net_income', ('net income', ['NetIncomeLoss'])),
    ('operating_income', ('operating income', ['OperatingIncomeLoss'])),
    ('gross_profit', ('gross profit', ['GrossProfit'])),
    ('assets', ('total assets', ['Assets'])),
    ('liabilities', ('total liabilities', ['Liabilities'])),
    ('equity', ('stockholders equity', ['StockholdersEquity'])),
    ('cash', ('cash and cash equivalents', ['CashAndCashEquivalentsAtCarryingValue'])),
    ('operating_cash_flow', ('net cash from operating activities', ['NetCashProvidedByUsedInOperatingActivities'])),
    ('investing_cash_flow', ('net cash from investing activities', ['NetCashProvidedByUsedInInvestingActivities'])),
    ('financing_cash_flow', ('net cash from financing activities', ['NetCashProvidedByUsedInFinancingActivities'])),
    ('current_assets', ('current assets', ['AssetsCurrent'])),
    ('current_liabilities', ('current liabilities', ['LiabilitiesCurrent'])),
    ('cost_of_revenue', ('cost of revenue', ['CostOfRevenue','CostOfGoodsAndServicesSold']))
])
INSTANT = {'assets','liabilities','equity','cash','current_assets','current_liabilities'}

session = requests.Session()
session.headers.update({'User-Agent':f'ClaimCheck Academic Research {EMAIL}',
                        'Accept-Encoding':'identity'})

def digest(data):
    return hashlib.sha256(data).hexdigest()

def fetch_snapshot(url, filename):
    """Read immutable local snapshots; download once when absent."""
    path = CACHE / filename
    if path.exists():
        raw = path.read_bytes()
    else:
        if not EMAIL or '@' not in EMAIL:
            raise ValueError('Set SEC_CONTACT_EMAIL to a real contact email before downloading SEC data.')
        for attempt in range(4):
            try:
                response = session.get(url, timeout=90)
                response.raise_for_status()
                raw = response.content
                json.loads(raw)
                path.write_bytes(raw)
                time.sleep(0.15)
                break
            except (requests.RequestException, ValueError):
                if attempt == 3:
                    raise
                time.sleep(2 ** attempt)
    return json.loads(raw), digest(raw)

def norm(value):
    words = re.sub(r'[^a-z0-9]+', ' ', str(value).casefold().replace('&',' and ')).split()
    return ' '.join(w for w in words if w not in
                    {'the','inc','incorporated','corp','corporation','co','company',
                     'ltd','plc','holding','holdings'})

def docnorm(value):
    return re.sub(r'[^a-z0-9]', '', str(value).casefold())

def cik10(value):
    return str(int(str(value).replace('.0',''))).zfill(10)

# Existing screening outputs are reused. New candidates are screened only once.
RAG = DATA / 'RAGBench'
FINANCE_META = DATA / 'FinanceBench' / 'financebench_document_information.jsonl'
AUDIT_PATH = OUT / 'combined_company_audit.csv'
MATCHES_PATH = OUT / 'ragbench_company_matches_review.csv'
if not FINANCE_META.exists():
    raise FileNotFoundError(f'Missing FinanceBench metadata: {FINANCE_META}')
rag_files = []
for subset in ('FinQA', 'TATQA'):
    for split in ('train', 'validation', 'test'):
        path = RAG / subset / f'{split}.csv'
        if not path.exists():
            raise FileNotFoundError(f'Missing processed RAGBench CSV: {path}')
        rag_files.append((subset, split, path))

# Add established SEC-listed companies to the original candidate pool.
# Ticker-to-CIK values are resolved from the SEC official ticker file.
EXTRA_TICKERS = '''ABT ABBV ACN ADP AEP AFL AIG ALL AMGN AON APD APH BDX BK BLK
BMY BSX C CB CCI CI CL CMCSA CME CMI COP CPB CVS DE DHR DOW DUK ECL
EL EMR EQIX ETN EXC FDX GD GILD GIS GM GPN HCA HLT HUM ICE ITW JCI
JNJ K KDP KHC KMB LLY LMT LULU MAR MDT MET MMC MMM MO MPC MRK MSI
NEM NOC NSC O OXY PAYX PFE PG PGR PH PNC PPG PRU PSX REGN ROK ROP
RSG RTX SBUX SCHW SHW SO SPGI SRE STT STZ SYK TGT TJX TMO TRV TSLA
UNH UNP UPS USB VLO VMC WM WY XOM YUM ZBH ZTS'''.split()

# Load the earlier audit without rescanning its original 50 companies.
if AUDIT_PATH.exists():
    audit_old = pd.read_csv(AUDIT_PATH, dtype={'cik':str}).fillna('')
    audit_old['cik'] = audit_old['cik'].map(cik10)
    if not {'ticker','cik','sec_name','financebench_match','ragbench_matches','status'}.issubset(audit_old.columns):
        raise ValueError('Existing audit columns are invalid.')
else:
    raise FileNotFoundError(f'Existing company audit is required: {AUDIT_PATH}')

metadata = pd.read_json(FINANCE_META, lines=True)
if not {'company','doc_name'}.issubset(metadata.columns):
    raise ValueError('FinanceBench metadata must contain company and doc_name.')
pdf_list, _ = fetch_snapshot('https://api.github.com/repos/patronus-ai/financebench/contents/pdfs', 'financebench_pdf_listing.json')
finance_names = set(metadata['company'].dropna().astype(str))
known_docs = {docnorm(x) for x in metadata['doc_name'].dropna()}
unresolved = []
for entry in pdf_list:
    filename = entry.get('name', '')
    if not filename.lower().endswith('.pdf') or docnorm(filename[:-4]) in known_docs:
        continue
    names = [company for prefix, company in PDF_ALIASES.items() if docnorm(filename).startswith(prefix)]
    if len(names) == 1:
        finance_names.add(names[0])
    else:
        unresolved.append(filename)
if unresolved:
    raise ValueError('Unresolved FinanceBench PDF names: ' + ', '.join(unresolved))
finance_keys = {norm(x) for x in finance_names}

sec_tickers, _ = fetch_snapshot('https://www.sec.gov/files/company_tickers.json', 'sec_company_tickers.json')
sec_index = {v['ticker'].upper(): (cik10(v['cik_str']), v['title']) for v in sec_tickers.values()}
for ticker in EXTRA_TICKERS:
    if ticker in sec_index and ticker not in CANDIDATES:
        CANDIDATES[ticker] = sec_index[ticker][0]

# Keep old results intact; screen only candidates missing from the audit.
old_tickers = set(audit_old['ticker'])
new_tickers = []  # v2 uses the completed audit; never screen additional companies
new_info = {}
for ticker in new_tickers:
    cik = CANDIDATES[ticker]
    try:
        info, hash_value = fetch_snapshot(f'https://data.sec.gov/submissions/CIK{cik}.json', f'submissions_{cik}.json')
        if cik10(info['cik']) != cik:
            raise ValueError('CIK mismatch')
        new_info[ticker] = (info, hash_value)
    except Exception as exc:
        print(f'Cannot resolve {ticker}: {exc}')
        new_info[ticker] = (None, '')

# Screen company names against the already prepared CSV files in one pass.
# A match is conservatively excluded, even if it is only a company mention.
# This avoids making unsupported claims that a matched company is safe.
patterns = {}
for ticker, (info, _) in new_info.items():
    if not info:
        continue
    name = info.get('name', '')
    aliases = [name, sec_index.get(ticker, ('',''))[1]]
    normalized = {norm(x) for x in aliases if x}
    safe = [x for x in normalized if len(x.replace(' ', '')) >= 7 and len(x.split()) >= 2]
    if safe:
        patterns[ticker] = [re.compile(r'(?<![a-z0-9])' + r'\s+'.join(map(re.escape, n.split())) + r'(?![a-z0-9])', re.I)
                            for n in safe]

new_hits = {ticker: 0 for ticker in new_tickers}
if patterns:
    for subset, split, path in rag_files:
        frame = pd.read_csv(path, usecols=lambda c: c in {'question','documents','response'},
                            dtype=str, keep_default_na=False)
        combined = frame.fillna('').agg(' '.join, axis=1)
        for ticker, regexes in patterns.items():
            hit_mask = pd.Series(False, index=combined.index)
            for regex in regexes:
                hit_mask |= combined.str.contains(regex, regex=True, na=False)
            new_hits[ticker] += int(hit_mask.sum())
        print(f'Screened new candidate names: {subset}/{split} ({len(frame)} rows)')

new_rows = []
for ticker in new_tickers:
    info, _ = new_info[ticker]
    cik = CANDIDATES[ticker]
    name = info.get('name','') if info else ''
    fin_match = bool(info and (norm(name) in finance_keys or norm(sec_index.get(ticker, ('',''))[1]) in finance_keys))
    count = new_hits[ticker]
    status = ('Needs Review' if not info else 'Excluded FinanceBench' if fin_match
              else 'RAGBench Match Review' if count else 'Eligible')
    new_rows.append(dict(ticker=ticker,cik=cik,sec_name=name,financebench_match=fin_match,
                         ragbench_matches=count,status=status))

audit = pd.concat([audit_old, pd.DataFrame(new_rows)], ignore_index=True)
audit = audit.drop_duplicates('ticker',keep='first').sort_values('cik').reset_index(drop=True)
audit.to_csv(AUDIT_PATH, index=False, encoding='utf-8')
eligible = audit[audit.status.eq('Eligible')].copy()
eligible['cik'] = eligible['cik'].map(cik10)
if eligible['cik'].duplicated().any():
    raise ValueError('Eligible company CIK values are not unique.')
eligible_by_cik = eligible.set_index('cik', drop=False)
eligible.to_csv(OUT/'eligible_companies.csv',index=False,encoding='utf-8')
print('Company screening:',audit.groupby('status').size().to_dict())
if len(eligible) < 40:
    raise RuntimeError(f'Only {len(eligible)} eligible companies after expansion; need 40. '
                       'Do not weaken overlap exclusions; extend EXTRA_TICKERS instead.')

# Load only the 40 locked companies; never choose new ones.
locked = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False)
required_lock_cols = {'ticker','cik','sec_name','split','dataset','claims_per_company'}
if not required_lock_cols.issubset(locked.columns) or len(locked) != 40:
    raise ValueError('Existing locked company file is missing columns or is not 40 rows.')
locked['cik'] = locked['cik'].map(cik10)
if locked.cik.nunique() != 40 or not set(locked.cik).issubset(set(eligible.cik)):
    raise ValueError('Locked companies are duplicated or no longer in the completed audit.')
sec_info = {}
for record in locked.to_dict('records'):
    cik, ticker = record['cik'], record['ticker']
    sec_info[ticker] = fetch_snapshot(f'https://data.sec.gov/submissions/CIK{cik}.json', f'submissions_{cik}.json')

# Extract only 10-K annual values for the selected filing and fiscal period.
def is_annual(entry, report_end, accession, instant):
    if entry.get('form') != '10-K' or entry.get('accn') != accession or entry.get('end') != report_end:
        return False
    if instant:
        return not entry.get('start')
    try:
        days = (date.fromisoformat(report_end) - date.fromisoformat(entry['start'])).days
        return 300 <= days <= 400
    except (KeyError,TypeError,ValueError):
        return False

def extract_metrics(facts, report_end, accession):
    result = {}
    gaap = facts.get('facts',{}).get('us-gaap',{})
    for metric,(label,tags) in METRICS.items():
        for tag in tags:
            entries = gaap.get(tag,{}).get('units',{}).get('USD',[])
            candidates = [x for x in entries if is_annual(x,report_end,accession,metric in INSTANT)
                          and isinstance(x.get('val'),(int,float))]
            if candidates:
                x = sorted(candidates,key=lambda z:(z.get('filed',''),str(z.get('frame',''))),reverse=True)[0]
                result[metric] = dict(metric=metric,metric_name=label,tag=tag,
                                      value=int(x['val']),start=x.get('start',''),
                                      end=report_end,accn=accession,filed=x.get('filed',''))
                break
    return result

def previous_value(facts, metric, end):
    entries = facts.get('facts',{}).get('us-gaap',{}).get(metric['tag'],{}).get('units',{}).get('USD',[])
    valid = []
    for x in entries:
        if x.get('form') != '10-K' or not x.get('end') or x['end'] >= end:
            continue
        try:
            gap = (date.fromisoformat(end)-date.fromisoformat(x['end'])).days
            if not 330 <= gap <= 400:
                continue
            if not is_annual(x,x['end'],x.get('accn'),metric['metric'] in INSTANT):
                continue
        except (TypeError,ValueError):
            continue
        if isinstance(x.get('val'),(int,float)):
            valid.append(x)
    return sorted(valid,key=lambda x:(x['end'],x.get('filed','')),reverse=True)[0] if valid else None

def amount(value):
    return f'{value / 1_000_000:,.3f}'

def transpose_digits(value):
    """Swap two unequal adjacent digits, preserving sign and magnitude order."""
    digits = list(str(abs(int(value))))
    for i in range(1, len(digits) - 1):
        if digits[i] != digits[i + 1] and digits[i + 1] != '0':
            digits[i], digits[i + 1] = digits[i + 1], digits[i]
            changed = int(''.join(digits)) * (1 if value >= 0 else -1)
            if changed != value:
                return changed
            digits[i], digits[i + 1] = digits[i + 1], digits[i]
    return None


def generate_company(record):
    ticker, cik, company, split = (record['ticker'],record['cik'],record['sec_name'],record['split'])
    submission, submission_hash = sec_info[ticker]
    recent = submission.get('filings',{}).get('recent',{})
    filings = []
    for form,accn,report,filed,primary in zip(
            recent.get('form',[]),recent.get('accessionNumber',[]),
            recent.get('reportDate',[]),recent.get('filingDate',[]),recent.get('primaryDocument',[])):
        if form == '10-K' and report and primary:
            filings.append((filed,accn,report,primary))
    if not filings:
        raise ValueError('No 10-K in cached SEC submissions')
    filed, accession, end, primary = max(filings)
    facts, facts_hash = fetch_snapshot(
        f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json',f'companyfacts_{cik}.json')
    metrics = extract_metrics(facts,end,accession)
    if len(metrics) < 8:
        raise ValueError(f'Only {len(metrics)} annual metrics available; need at least 8')
    filing_url = f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession.replace("-", "")}/{primary}'
    pools = {k:[] for k in QUOTAS}
    for key, m in metrics.items():
        actual = m['value']
        if abs(actual) < 1_000_000:
            continue
        common = dict(ticker=ticker,cik=cik,sec_name=company,split=split,
                      metric=key,xbrl_tag=m['tag'],actual_value_usd=actual,
                      report_end=end,accession=accession,filing_date=filed,
                      filing_url=filing_url,source_facts_sha256=facts_hash,
                      source_submissions_sha256=submission_hash,
                      evidence_start=m['start'],evidence_end=end,evidence_value_usd=actual,
                      evidence_tag=m['tag'],evidence_unit='USD')
        prefix = f"{company}'s {m['metric_name']} for the fiscal period ended {end}"
        pools['Supported'].append(dict(common,label='Supported',error_type='Supported',
            claim=f'{prefix} was USD {amount(actual)} million.',
            claimed_value_usd=actual,reference_value_usd=actual))

        # A controlled 1%-5% error, large enough to survive 3-decimal million rounding.
        pct = 0.01 + (int(hashlib.sha256(f'{SEED}:{cik}:{key}'.encode()).hexdigest()[:8], 16) % 401) / 10000
        changed = actual + (max(round(abs(actual) * pct), 1) * (1 if actual >= 0 else -1))
        if round(changed/1e6,3) != round(actual/1e6,3):
            pools['Small Numerical'].append(dict(common,label='Unsupported',error_type='Small Numerical',
                claim=f'{prefix} was USD {amount(changed)} million.',
                claimed_value_usd=changed,reference_value_usd=actual))

        # Alternate digit transpositions and sign flips across companies/metrics.
        digit_changed = transpose_digits(actual)
        sign_changed = -actual
        use_digit = int(hashlib.sha256(f'{cik}:{key}'.encode()).hexdigest()[:2],16) % 2 == 0
        variant = [('Transposed Digits', digit_changed), ('Flipped Sign', sign_changed)] if use_digit else [('Flipped Sign', sign_changed), ('Transposed Digits', digit_changed)]
        for error_type, wrong in variant:
            if wrong is not None and wrong != actual and round(wrong/1e6,3) != round(actual/1e6,3):
                pools['Digit or Sign'].append(dict(common,label='Unsupported',error_type=error_type,
                    claim=f'{prefix} was USD {amount(wrong)} million.',
                    claimed_value_usd=wrong,reference_value_usd=actual))
                break

        prior = previous_value(facts,m,end)
        if prior and int(prior['val']) != actual:
            old_end = prior['end']
            pools['Temporal'].append(dict(common,label='Unsupported',error_type='Temporal',
                claim=f"{company}'s {m['metric_name']} for the fiscal period ended {old_end} was USD {amount(actual)} million.",
                claimed_value_usd=actual,reference_value_usd=int(prior['val']),
                comparison_period_end=old_end,comparison_accession=prior.get('accn',''),
                comparison_value_usd=int(prior['val']),
                evidence_value_usd=int(prior['val']),evidence_end=old_end,
                evidence_start=prior.get('start',''),evidence_tag=m['tag'],
                evidence_accession=prior.get('accn','')))
        pools['Unit'].append(dict(common,label='Unsupported',error_type='Unit',
            claim=f'{prefix} was USD {amount(actual)} billion.',
            claimed_value_usd=actual*1000,reference_value_usd=actual))
        for other_key, other in metrics.items():
            if key == other_key or abs(actual-other['value']) < 1_000_000:
                continue
            pools['Metric Confusion'].append(dict(common,label='Unsupported',error_type='Metric Confusion',
                claim=f"{company}'s {other['metric_name']} for the fiscal period ended {end} was USD {amount(actual)} million.",
                claimed_value_usd=actual,reference_value_usd=other['value'],
                claimed_metric=other_key,comparison_tag=other['tag'],
                evidence_value_usd=other['value'],evidence_tag=other['tag']))
    rng = random.Random(f'{SEED}:{cik}')
    rows = []
    for kind, quota in QUOTAS.items():
        pool = pools[kind]
        rng.shuffle(pool)
        distinct, seen = [], set()
        for row in pool:
            if row['metric'] not in seen:
                distinct.append(row)
                seen.add(row['metric'])
        if len(distinct) < quota:
            raise ValueError(f'Insufficient {kind} examples for {ticker}: {len(distinct)} of {quota}')
        rows.extend(distinct[:quota])
    if len(rows) != 10 or len({x['claim'] for x in rows}) != 10:
        raise ValueError('Company claims are not 10 unique statements')
    for index,row in enumerate(rows,1):
        row['claim_id'] = f'SEC-{ticker}-{index:02d}'
        row['source_type'] = 'SEC 10-K XBRL companyfacts'
        row['verification_status'] = 'rule_checked_pending_original_filing_review'
    return rows


# Fully automatic Storyboard B extraction from SEC inline XBRL 10-K filings.
# No manual review CSV. Only facts whose context, USD unit, fiscal period and
# consolidation comparator can be checked programmatically are accepted.
from bs4 import BeautifulSoup
from decimal import Decimal, InvalidOperation
from datetime import datetime

SEGMENT_DIMENSIONS = ('statementbusinesssegmentsaxis', 'operatingsegmentsaxis',
                      'reportablesegmentsaxis', 'businesssegmentsaxis')
REVENUE_TAGS = {'revenuefromcontractwithcustomerexcludingassessedtax',
                'revenuefromcontractwithcustomerincludingassessedtax',
                'revenues', 'salesrevenuenet', 'salesrevenuegoodsnet',
                'segmentrevenue', 'revenue'}

def localname(name):
    return str(name).split(':')[-1].casefold()

def xml_attr(tag, key):
    for k, v in tag.attrs.items():
        if localname(k) == key.casefold():
            return v[0] if isinstance(v,list) else v
    return None

def parse_usd(raw, scale, sign):
    txt = str(raw).replace(',', '').replace('\u2212','-').strip()
    txt = re.sub(r'[^0-9.()\-]', '', txt)
    if not txt or txt in ('-', '.', '()'):
        return None
    if txt.startswith('(') and txt.endswith(')'):
        txt = '-' + txt[1:-1]
    try:
        value = Decimal(txt) * (Decimal(10) ** int(scale or 0))
        if str(sign or '').strip() == '-':
            value = -abs(value)
        return int(value) if value == value.to_integral_value() and value > 0 else None
    except (InvalidOperation, ValueError, OverflowError):
        return None

def filing_candidates(ticker, cik):
    info, _ = sec_info[ticker]
    recent = info.get('filings',{}).get('recent',{})
    entries=[]
    for form,accn,end,filed,primary in zip(
        recent.get('form',[]),recent.get('accessionNumber',[]),
        recent.get('reportDate',[]),recent.get('filingDate',[]),
        recent.get('primaryDocument',[])):
        if form == '10-K' and end and primary:
            url = f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accn.replace("-", "")}/{primary}'
            entries.append((filed,accn,end,url))
    return sorted(entries,reverse=True)

def download_filing(url, cik, accession):
    path = CACHE / f'filing_{cik}_{accession.replace("-", "")}.html'
    if path.exists():
        return path.read_bytes(), digest(path.read_bytes())
    if not EMAIL or '@' not in EMAIL:
        raise ValueError('Set SEC_CONTACT_EMAIL to your real email for SEC access.')
    for attempt in range(4):
        try:
            r = session.get(url,timeout=120)
            r.raise_for_status()
            raw = r.content
            if len(raw) < 1000:
                raise ValueError('Filing response unexpectedly short')
            path.write_bytes(raw)
            time.sleep(0.2)
            return raw, digest(raw)
        except (requests.RequestException,ValueError):
            if attempt == 3:
                raise
            time.sleep(2**attempt)

def extract_segment_facts(raw, ticker, cik, accession, report_end, url, source_hash):
    soup = BeautifulSoup(raw, 'lxml')
    contexts={}
    for tag in soup.find_all(lambda t: localname(t.name)=='context'):
        cid = xml_attr(tag,'id')
        if not cid:
            continue
        start = tag.find(lambda t: localname(t.name)=='startdate')
        end = tag.find(lambda t: localname(t.name)=='enddate')
        if not start or not end:
            continue
        start_text, end_text = start.get_text(strip=True), end.get_text(strip=True)
        try:
            days=(date.fromisoformat(end_text)-date.fromisoformat(start_text)).days
        except ValueError:
            continue
        if not 300 <= days <= 400 or end_text != report_end:
            continue
        members=[]
        for m in tag.find_all(lambda t: localname(t.name)=='explicitmember'):
            axis=str(xml_attr(m,'dimension') or '').split(':')[-1].casefold()
            member=m.get_text(strip=True).split(':')[-1]
            members.append((axis,member))
        seg=[member for axis,member in members if any(d in axis for d in SEGMENT_DIMENSIONS)]
        if len(seg)!=1 or len(members)!=1:
            continue
        contexts[str(cid)] = (seg[0],start_text,end_text)
    units={}
    for unit in soup.find_all(lambda t: localname(t.name)=='unit'):
        uid=xml_attr(unit,'id')
        measures=[m.get_text(strip=True).casefold() for m in unit.find_all(lambda t: localname(t.name)=='measure')]
        if uid:
            units[str(uid)] = len(measures)==1 and measures[0].split(':')[-1]=='usd'
    facts=[]
    for tag in soup.find_all(lambda t: localname(t.name) in ('nonfraction','non-fraction')):
        ctx=str(xml_attr(tag,'contextref') or '')
        if ctx not in contexts:
            continue
        metric=str(xml_attr(tag,'name') or '').split(':')[-1].casefold()
        if metric not in REVENUE_TAGS:
            continue
        if not units.get(str(xml_attr(tag,'unitref') or ''),False):
            continue
        amount_usd=parse_usd(tag.get_text(' ',strip=True),xml_attr(tag,'scale'),xml_attr(tag,'sign'))
        if not amount_usd:
            continue
        segment,start,end=contexts[ctx]
        facts.append(dict(ticker=ticker,cik=cik,accession=accession,
                          filing_url=url,source_sha256=source_hash,
                          fiscal_period=end,period_start=start,
                          segment_name=segment,segment_revenue_usd=amount_usd,
                          revenue_tag=metric,context_id=ctx))
    return facts

def consolidated_revenue(cik, period_end):
    facts, _ = fetch_snapshot(
        f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json',
        f'companyfacts_{cik}.json')
    gaap=facts.get('facts',{}).get('us-gaap',{})
    for tag in METRICS['revenue'][1]:
        entries=gaap.get(tag,{}).get('units',{}).get('USD',[])
        candidates=[]
        for x in entries:
            if x.get('form')!='10-K' or x.get('end')!=period_end:
                continue
            try:
                days=(date.fromisoformat(x['end'])-date.fromisoformat(x['start'])).days
            except (KeyError,ValueError,TypeError):
                continue
            if 300<=days<=400 and isinstance(x.get('val'),(int,float)) and x['val']>0:
                candidates.append(x)
        if candidates:
            best=sorted(candidates,key=lambda x:(x.get('filed',''),x.get('accn','')),reverse=True)[0]
            return int(best['val']),tag,best.get('accn','')
    return None,None,None

# A Storyboard company needs at least 5 distinct segment-period observations.
# Keep the exact source and consolidated comparator for every generated claim.
by_cik={}
segment_failures=[]
for record in locked.loc[locked.dataset.eq('storyboard_b')].sort_values('cik').to_dict('records'):
    ticker,cik=record['ticker'],record['cik']
    facts=[]
    try:
        for _,accession,report_end,url in filing_candidates(ticker,cik)[:5]:
            raw,source_hash=download_filing(url,cik,accession)
            extracted=extract_segment_facts(raw,ticker,cik,accession,report_end,url,source_hash)
            total,total_tag,total_accn=consolidated_revenue(cik,report_end)
            if total is None:
                continue
            for fact in extracted:
                if fact['segment_revenue_usd'] >= total:
                    continue
                fact.update(total_revenue_usd=total,total_revenue_tag=total_tag,
                            total_revenue_accession=total_accn)
                facts.append(fact)
            if len(facts)>=5:
                break
        facts=sorted(facts,key=lambda f:(f['fiscal_period'],f['segment_name'],
                                         f['segment_revenue_usd'],f['revenue_tag'],f['context_id']))
        unique={}
        for f in facts:
            key=(f['fiscal_period'],f['segment_name'],f['segment_revenue_usd'])
            unique.setdefault(key,f)
        selected=list(unique.values())
        if len(selected)>=5:
            by_cik[cik]=selected[:5]
            print(f'Segment evidence {len(by_cik)}/10: {ticker} ({len(selected)} facts)')
        else:
            segment_failures.append(dict(ticker=ticker,cik=cik,reason=f'Only {len(selected)} distinct segment facts'))
    except Exception as exc:
        segment_failures.append(dict(ticker=ticker,cik=cik,reason=str(exc)))
    # All locked Storyboard companies must be checked.
if len(by_cik)<10:
    pd.DataFrame(segment_failures).to_csv(OUT/'storyboard_automatic_extraction_failures.csv',index=False)
    raise RuntimeError(f'Automatic SEC extraction found {len(by_cik)} qualifying Storyboard companies; '
                       'need 10. See storyboard_automatic_extraction_failures.csv. '
                       'No manual data entry and no fabricated facts.')
# Deterministic selection: Storyboard first, General from separate companies.
# Do not reuse an old 20-company lock. New lock is written only after full success.
if MANIFEST.exists():
    chosen = pd.read_csv(MANIFEST,dtype=str,keep_default_na=False)
    required = {'ticker','cik','sec_name','split','dataset','claims_per_company'}
    if not required.issubset(chosen.columns):
        raise ValueError('Existing 40-company lock has missing columns.')
    chosen['cik'] = chosen.cik.map(cik10)
    if chosen.cik.nunique() != 40 or len(chosen) != 40:
        raise ValueError('Existing lock must contain 40 distinct companies.')
    if not set(chosen.cik).issubset(set(eligible.cik)):
        raise ValueError('Existing lock contains ineligible companies.')
    for dataset, expected in [('general',dict(SPLIT_COMPANIES)),('storyboard_b',STORY_COMPANIES)]:
        actual = chosen[chosen.dataset.eq(dataset)].groupby('split').size().to_dict()
        if actual != expected:
            raise ValueError(f'Invalid locked split counts for {dataset}: {actual}')
    if not set(chosen.loc[chosen.dataset.eq('storyboard_b'),'cik']).issubset(by_cik):
        raise ValueError('Locked Storyboard company lacks 5 automatically extracted segment facts.')
    print('Reusing existing 40-company lock.')

# Generate from locked companies, with identical deterministic ordering.
general_rows = []
for record in chosen.loc[chosen.dataset.eq('general')].to_dict('records'):
    general_rows.extend(generate_company(record))
claims = pd.DataFrame(general_rows)
assert len(claims) == 300 and claims.claim.nunique() == 300
assert claims.groupby('split').size().to_dict() == {'train':210,'validation':60,'test':30}
assert claims.groupby('cik').size().eq(10).all()
assert claims.label.value_counts().to_dict() == {'Supported':150,'Unsupported':150}
assert claims.groupby('cik').split.nunique().eq(1).all()

storyboard_rows = []
for record in chosen.loc[chosen.dataset.eq('storyboard_b')].to_dict('records'):
    facts = by_cik[record['cik']]
    for index,fact in enumerate(facts,1):
        segment = fact['segment_name'].strip()
        period = fact['fiscal_period']
        amount_usd = int(fact['segment_revenue_usd'])
        amount_str = f'{amount_usd:,}'
        common = dict(ticker=record['ticker'],cik=record['cik'],
                      sec_name=record['sec_name'],split=record['split'],
                      segment_name=segment,report_end=period,
                      source_type='SEC 10-K inline XBRL automatic extraction',
                      filing_url=fact['filing_url'],accession=fact['accession'],
                      segment_revenue_usd=amount_usd,
                      total_revenue_usd=int(fact['total_revenue_usd']),
                      segment_context_id=fact['context_id'],
                      segment_xbrl_tag=fact['revenue_tag'],
                      total_xbrl_tag=fact['total_revenue_tag'],
                      source_sha256=fact['source_sha256'],
                      verification_status='automated_xbrl_context_checked')
        storyboard_rows.append(dict(common,claim_id=f'SB-{record["ticker"]}-{index:02d}',
            label='Unsupported',error_type='Segment vs Total Revenue',
            claim=f"{record['sec_name']}'s {segment} revenue for the fiscal period ended {period} represented the company's entire revenue of USD {amount_str}.",
            claimed_value_usd=amount_usd,reference_value_usd=int(fact['total_revenue_usd'])))
        if index <= STORY_SUPPORTED_PER_COMPANY:
            storyboard_rows.append(dict(common,claim_id=f'SB-{record["ticker"]}-S{index:02d}',
                label='Supported',error_type='Supported Segment Revenue',
                claim=f"{record['sec_name']}'s {segment} segment revenue for the fiscal period ended {period} was USD {amount_str}.",
                claimed_value_usd=amount_usd,reference_value_usd=amount_usd))
storyboard = pd.DataFrame(storyboard_rows)
assert len(storyboard) == 100 and storyboard.claim.nunique() == 100
assert storyboard.label.value_counts().to_dict() == {'Unsupported':50,'Supported':50}
assert storyboard.groupby('split').size().to_dict() == STORYBOARD_COUNTS
assert storyboard.groupby('cik').size().eq(10).all()
assert storyboard.groupby(['cik','label']).size().eq(5).all()
assert storyboard.groupby('cik').split.nunique().eq(1).all()
assert set(claims.cik).isdisjoint(set(storyboard.cik))
assert set(claims.claim).isdisjoint(set(storyboard.claim))
assert chosen.groupby('cik').split.nunique().eq(1).all()
assert chosen.cik.nunique() == 40
assert claims.groupby(['cik','label']).size().eq(5).all()

# Publish only after all validations pass. Old 20-company lock is left untouched.
claims = claims.sort_values(['split','cik','claim_id'],kind='stable').reset_index(drop=True)
storyboard = storyboard.sort_values(['split','cik','claim_id'],kind='stable').reset_index(drop=True)
output_frames = {'general_claims_300.csv':claims,'storyboard_b_100.csv':storyboard}
for split in SPLIT_COMPANIES:
    output_frames[f'general_claims_{split}.csv'] = claims.loc[claims.split.eq(split)]
    output_frames[f'storyboard_b_{split}.csv'] = storyboard.loc[storyboard.split.eq(split)]

# The lock is written after validation; all files use fixed ordering and newline handling.
chosen = chosen.sort_values(['dataset','split','cik'],kind='stable').reset_index(drop=True)
# Preserve the existing locked company file byte-for-byte.
for filename,frame in output_frames.items():
    frame.to_csv(OUT/filename,index=False,encoding='utf-8',lineterminator='\n')

all_files = [MANIFEST,*(OUT/name for name in output_frames)]
checksums = [{'file':p.name,'sha256':digest(p.read_bytes())} for p in all_files]
pd.DataFrame(checksums).to_csv(OUT/'output_sha256_400.csv',index=False,lineterminator='\n')
input_files = [FINANCE_META,*(path for _,_,path in rag_files)]
source_files = sorted(CACHE.glob('*.json')) + sorted(CACHE.glob('filing_*.html'))
manifest = {
    'format':'ClaimCheck Inserted Error Set v2, 40 companies, 400 claims',
    'seed':SEED,'company_count':40,'general_company_count':30,
    'storyboard_company_count':10,'general_claim_count':300,
    'storyboard_claim_count':100,'storyboard_label_counts':{'Unsupported':50,'Supported':50},
    'general_label_counts':{'Unsupported':150,'Supported':150},
    'total_label_counts':{'Unsupported':200,'Supported':200},
    'generation_quotas':dict(QUOTAS),
    'general_split_counts':{'train':210,'validation':60,'test':30},
    'storyboard_split_counts':STORYBOARD_COUNTS,
    'review_note':'Automatic inline XBRL extraction; verify selected filing context and segment definitions before publication',
    'input_sha256':{str(p.relative_to(PROJECT)):digest(p.read_bytes()) for p in input_files},
    'snapshot_sha256':{str(p.relative_to(PROJECT)):digest(p.read_bytes()) for p in source_files},
    'output_sha256':{p.name:digest(p.read_bytes()) for p in [MANIFEST,*(OUT/name for name in output_frames)]},
    'python_packages':{'pandas':pd.__version__,'requests':requests.__version__},
}
(OUT/'reproducibility_manifest_400.json').write_text(
    json.dumps(manifest,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print('SUCCESS: 300 General + 100 Storyboard B claims, 40 disjoint companies.')
print('General:',claims.groupby('split').size().to_dict())
print('Storyboard:',storyboard.groupby(['split','label']).size().to_dict())
print('Output:',OUT)

Company screening: {'Eligible': 102, 'Excluded FinanceBench': 22, 'RAGBench Match Review': 35}


/var/folders/c_/9lbsszj52z9f_qw3y9llbtw80000gn/T/ipykernel_27120/2464365625.py:523: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(raw, 'lxml')


Segment evidence 1/10: CPB (6 facts)
Segment evidence 2/10: CAT (5 facts)
Segment evidence 3/10: CMI (5 facts)
Segment evidence 4/10: ECL (8 facts)
Segment evidence 5/10: KMB (8 facts)
Segment evidence 6/10: PH (6 facts)
Segment evidence 7/10: PG (6 facts)
Segment evidence 8/10: SHW (6 facts)
Segment evidence 9/10: SO (6 facts)
Segment evidence 10/10: WY (6 facts)
Reusing existing 40-company lock.
SUCCESS: 300 General + 100 Storyboard B claims, 40 disjoint companies.
General: {'test': 30, 'train': 210, 'validation': 60}
Storyboard: {('test', 'Supported'): 25, ('test', 'Unsupported'): 25, ('train', 'Supported'): 15, ('train', 'Unsupported'): 15, ('validation', 'Supported'): 10, ('validation', 'Unsupported'): 10}
Output: /Users/stephanie/Documents/python/Claimcheck/Data/Inserted Error Set


In [4]:
import pandas as pd
from pathlib import Path

# Automatically locate the Claimcheck project
cwd = Path.cwd().resolve()

PROJECT = next(
    (p for p in (cwd, *cwd.parents)
     if p.name.casefold() == "claimcheck"),
    None
)

if PROJECT is None:
    raise RuntimeError("Cannot find Claimcheck project directory.")

OUT = PROJECT / "Data" / "Inserted Error Set"

# Load datasets
general = pd.read_csv(OUT / "general_claims_300.csv")
storyboard = pd.read_csv(OUT / "storyboard_b_100.csv")

combined = pd.concat([general, storyboard], ignore_index=True)

# Check dataset
print("Total claims:", len(combined))
print("Unique companies:", combined["cik"].nunique())

print("\nLabel distribution:")
print(combined["label"].value_counts())

print("\nError types:")
print(combined["error_type"].value_counts())

print("\nSplit distribution:")
print(pd.crosstab(combined["split"], combined["label"]))

Total claims: 400
Unique companies: 40

Label distribution:
label
Supported      200
Unsupported    200
Name: count, dtype: int64

Error types:
error_type
Supported                    150
Segment vs Total Revenue      50
Supported Segment Revenue     50
Small Numerical               30
Temporal                      30
Metric Confusion              30
Unit                          30
Flipped Sign                  18
Transposed Digits             12
Name: count, dtype: int64

Split distribution:
label       Supported  Unsupported
split                             
test               40           40
train             120          120
validation         40           40
